# AeroPulse Propagation — 02: Temporal & Station-Relative Features

Audit what the upstream feature store already provides, add the station
climatology it does not, and prove the whole feature block is past-only.

## Objective

Improvement plan sections 7-11 and 52 (Notebook 02).

The plan asks for cyclical time features, a PM2.5 lag hierarchy, rolling
statistics, station-relative baselines and multi-scale / wind-aware spatial
neighbours. Before building any of that, this notebook **checks what already
exists**, because the pm25 Phase-2 feature store was substantially extended
after the improvement plan was written. Re-deriving a column that is already
there is not neutral — it doubles the surface area for a leakage bug.

What this notebook actually does:

1. audits the upstream feature store against every item the plan lists
2. adds the genuinely missing block: **station-relative climatology**, fitted
   on the training window only
3. runs the past-only verification the plan requires (`test_rolling_features_use_past_only`)
4. runs a leakage scan over the full candidate feature set

In [1]:
# ============================================================================
# SETUP — everything reusable lives in propagation_toolkit.py, not in cells.
# ============================================================================
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import propagation_toolkit as pt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

cfg = pt.load_config()
HORIZONS = cfg.horizons
print("kernel:", sys.executable)

env: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast/.env
project: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast
horizons: [1, 3, 6, 12, 24, 48] | schema 2.0.0
kernel: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/.venv/bin/python


## 1. Load the validated base table

In [2]:
base_file = cfg.base_dir / "station_pm25_series.parquet"
if not base_file.exists():
    raise FileNotFoundError("Run 01_propagation_data_quality.ipynb first.")

df = pd.read_parquet(base_file)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df = df.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)
print("rows", f"{len(df):,}", "| columns", df.shape[1], "| stations", df["location_id"].nunique())

rows 1,705,252 | columns 208 | stations 149


## 2. Audit: what the plan asks for vs what already exists

Each row is a feature family the improvement plan requests. `present` counts
columns already in the store that satisfy it.

In [3]:
REQUESTED = {
    "S7 cyclical time":        ["sin_hour", "cos_hour", "sin_doy", "cos_doy"],
    "S7 cyclical day-of-week": ["sin_dow", "cos_dow"],
    "S8 rolling mean":         [f"pm25_roll_mean_{w}h" for w in (3, 6, 12, 24, 48)],
    "S8 rolling std":          [f"pm25_roll_std_{w}h" for w in (6, 12, 24)],
    "S8 rolling min/max":      ["pm25_roll_min_24h", "pm25_roll_max_24h"],
    "S8 slopes":               [f"pm25_slope_{w}h" for w in (3, 6, 12)],
    "S9 lag hierarchy":        [f"pm25_lag_{l}h" for l in (1, 2, 3, 6, 12, 24, 48, 72, 168)],
    "S10 station stats":       ["station_pm25_mean", "station_pm25_std", "station_pm25_p90", "station_pm25_p95"],
    "S10 station baselines":   ["station_baseline_by_hour", "station_baseline_by_month", "station_recent_anomaly"],
    "S11 multi-scale spatial": ["regional_pm25_mean_25km", "regional_pm25_mean_50km", "regional_pm25_mean_100km"],
    "S11 spatial dispersion":  ["regional_pm25_std_50km", "regional_pm25_max_50km"],
    "S11 wind-aware upwind":   ["upwind_pm25_mean", "upwind_pm25_max", "upwind_pm25_gradient"],
    "S14 fire by radius":      [f"fire_frp_sum_{r}km_24h" for r in (10, 25, 50)],
    "S14 upwind fire":         ["upwind_fire_frp_50km", "upwind_fire_count_50km", "upwind_fire_fraction_100km"],
}

rows = []
for family, columns in REQUESTED.items():
    have = [c for c in columns if c in df.columns]
    rows.append({
        "family": family,
        "requested": len(columns),
        "present": len(have),
        "status": "complete" if len(have) == len(columns) else ("partial" if have else "MISSING"),
        "missing": ", ".join(c for c in columns if c not in df.columns)[:70],
    })
audit = pd.DataFrame(rows)
print(audit.to_string(index=False))
print("\nfamilies already complete:", int((audit["status"] == "complete").sum()), "/", len(audit))

                 family  requested  present   status                                                                missing
       S7 cyclical time          4        4 complete                                                                       
S7 cyclical day-of-week          2        0  MISSING                                                       sin_dow, cos_dow
        S8 rolling mean          5        5 complete                                                                       
         S8 rolling std          3        3 complete                                                                       
     S8 rolling min/max          2        2 complete                                                                       
              S8 slopes          3        3 complete                                                                       
       S9 lag hierarchy          9        9 complete                                                                       
      S1

### Reading the audit

Everything the plan asks for in sections 7-9, 11 and 14 is already in the
store. `fire_frp_sum_25km_24h` is absent only because the upstream radii are
5/10/20/50/100/150 km rather than the plan's 10/25/50 — a naming difference,
not a capability gap.

The one genuinely missing family is **S10 station-relative features**, and
that is the block this notebook adds.

## 3. Cyclical time

Idempotent — the upstream store already has `sin_hour`/`cos_hour`/`sin_doy`/
`cos_doy`; this only fills in the day-of-week pair.

One convention has to be checked rather than assumed: the upstream `hour`,
`dow` and `doy` are **IST local time**, not UTC. That is the right choice for
PM2.5 (the diurnal cycle follows local traffic and boundary-layer collapse,
not Greenwich), but it means any new cyclical feature must be derived from the
same columns rather than re-derived from `timestamp_utc`.

In [4]:
utc_match = float((df["hour"] == df["timestamp_utc"].dt.hour).mean())
local_match = float((df["hour"] == pd.to_datetime(df["timestamp_local"]).dt.hour).mean())
print(f"upstream 'hour' equals UTC hour   : {utc_match:.1%}")
print(f"upstream 'hour' equals local hour : {local_match:.1%}")
assert local_match > 0.99, "expected IST-local calendar columns"

before = set(df.columns)
df = pt.add_cyclical_time(df)
print("added:", sorted(set(df.columns) - before))

# Verify against the same clock the upstream columns use, not against UTC.
consistent = float(
    np.isclose(df["sin_hour"], np.sin(2 * np.pi * df["hour"] / 24), atol=1e-5).mean()
)
print(f"sin_hour consistent with the 'hour' column: {consistent:.1%}")
print(f"sin_dow  consistent with the 'dow'  column: "
      f"{float(np.isclose(df['sin_dow'], np.sin(2 * np.pi * df['dow'] / 7), atol=1e-5).mean()):.1%}")

upstream 'hour' equals UTC hour   : 0.0%
upstream 'hour' equals local hour : 100.0%
added: ['cos_dow', 'sin_dow']
sin_hour consistent with the 'hour' column: 100.0%
sin_dow  consistent with the 'dow'  column: 100.0%


## 4. Station-relative climatology (plan section 10)

The subtle part is *when* these statistics are allowed to be computed. A
station mean fitted over the whole dataset embeds the test period's pollution
level into a training feature — the model learns "this station runs at
90 ug/m3" partly from the very rows it will be scored on.

`pt.StationBaselines` fits strictly before the training cutoff and stores the
cutoff in the artifact, so production recomputes it the same way. Unseen
stations fall back to the global hour-of-day climatology instead of NaN,
which is what makes the hierarchical fallback in the inference layer possible.

In [5]:
fit_end = df["timestamp_utc"].quantile(cfg.train_fraction)
print("baseline fit window:", df["timestamp_utc"].min(), "->", fit_end)
print("rows used for fitting:", f"{int((df['timestamp_utc'] < fit_end).sum()):,}")

t0 = time.time()
baselines = pt.StationBaselines.fit(df, fit_end)
df = pt.StationBaselines.transform(baselines, df)
print(f"fitted + applied in {time.time() - t0:.1f}s")
print(df[pt.FEATURE_COLUMNS_STATION].describe().T[["mean", "std", "min", "max"]].round(2).to_string())

baseline fit window: 2025-02-18 20:30:00+00:00 -> 2026-03-22 01:30:00+00:00
rows used for fitting: 1,193,612


fitted + applied in 0.3s
                            mean    std     min      max
station_pm25_mean          40.54  11.56    9.54    97.67
station_pm25_std           32.97  13.14    4.84    97.54
station_pm25_p90           78.27  27.53   26.30   228.00
station_pm25_p95           96.14  37.06   28.20   309.00
station_baseline_by_hour   33.27  10.95    2.05    96.60
station_baseline_by_month  33.29  20.06    0.00   214.50
pm25_vs_station_hour        3.29  32.95  -94.93  1830.20
pm25_vs_station_month       3.27  28.15 -189.00  1841.95
pm25_vs_station_mean       -3.98  33.38  -96.67  1821.41
station_recent_anomaly      0.09   0.96   -6.83   111.55
station_over_p90            0.08   0.27    0.00     1.00


### Leakage test: the baselines must not move when the future changes

In [6]:
tampered = df[["location_id", "timestamp_utc", "pm25", "hour"]].copy()
future = tampered["timestamp_utc"] >= fit_end
tampered.loc[future, "pm25"] = tampered.loc[future, "pm25"] * 10
refit = pt.StationBaselines.fit(tampered, fit_end)

identical = np.allclose(
    baselines.station["station_pm25_mean"].to_numpy(),
    refit.station["station_pm25_mean"].to_numpy(),
)
print("multiplied every post-cutoff PM2.5 by 10")
print("station_pm25_mean unchanged:", identical)
assert identical, "station baselines are leaking future data"

multiplied every post-cutoff PM2.5 by 10
station_pm25_mean unchanged: True


### Is the station-relative signal worth anything?

`pm25_vs_station_hour` should carry information the raw level does not: it
answers "is this unusual *for here, at this hour*" rather than "is this high".

In [7]:
sample = df.dropna(subset=["target_pm25_t_plus_6h"]).sample(200_000, random_state=42)
resid = sample["target_pm25_t_plus_6h"] - sample["pm25"]
for column in ["pm25", "pm25_vs_station_hour", "station_recent_anomaly", "pm25_vs_station_mean"]:
    corr = sample[column].corr(resid)
    print(f"  corr({column:<24}, 6h residual) = {corr:+.4f}")

  corr(pm25                    , 6h residual) = -0.4702
  corr(pm25_vs_station_hour    , 6h residual) = -0.4434
  corr(station_recent_anomaly  , 6h residual) = -0.4209
  corr(pm25_vs_station_mean    , 6h residual) = -0.4914


## 5. Past-only verification (plan section 47)

The plan requires `test_rolling_features_use_past_only`. This recomputes each
feature from the station's own history and asserts equality — a real check,
not an inspection of the column name.

In [8]:
checks = []
for column, builder, recipe in [
    ("pm25_lag_1h",       pt.time_lag(1),        "value observed exactly 1 h earlier"),
    ("pm25_lag_24h",      pt.time_lag(24),       "value observed exactly 24 h earlier"),
    ("pm25_lag_168h",     pt.time_lag(168),      "value observed exactly 168 h earlier"),
    ("pm25_roll_mean_6h", pt.rolling_past(6),    "mean over the previous 6 h, current row excluded"),
    ("pm25_roll_mean_24h",pt.rolling_past(24),   "mean over the previous 24 h, current row excluded"),
    ("pm25_roll_std_6h",  pt.rolling_past(6, "std"), "std over the previous 6 h"),
]:
    record = pt.assert_past_only(df, column, builder)
    record["recipe"] = recipe
    checks.append(record)
    print(f"PASS {column:<20} {recipe:<42} max|diff|={record['max_abs_diff']:.2e}")

# The distinction matters. A positional recipe would span the gaps found in 01,
# and it is not what the upstream store actually did.
positional = pt.assert_past_only(df, "pm25_lag_24h", lambda g: g["pm25"].shift(24), tol=np.inf)
positional_roll = pt.assert_past_only(
    df, "pm25_roll_mean_6h", lambda g: g["pm25"].rolling(6, min_periods=1).mean().shift(1), tol=np.inf
)
print(f"pm25_roll_mean_6h vs positional rolling(6).shift(1): max|diff| = {positional_roll['max_abs_diff']:.1f}")
print(f"\nsame column against a positional shift(24): max|diff| = {positional['max_abs_diff']:.1f}")
print("-> upstream lags are time-aligned; with 56k gaps the positional recipe would be wrong.")

PASS pm25_lag_1h          value observed exactly 1 h earlier         max|diff|=0.00e+00


PASS pm25_lag_24h         value observed exactly 24 h earlier        max|diff|=0.00e+00


PASS pm25_lag_168h        value observed exactly 168 h earlier       max|diff|=0.00e+00


PASS pm25_roll_mean_6h    mean over the previous 6 h, current row excluded max|diff|=2.44e-05


PASS pm25_roll_mean_24h   mean over the previous 24 h, current row excluded max|diff|=2.03e-05


PASS pm25_roll_std_6h     std over the previous 6 h                  max|diff|=3.15e-04


pm25_roll_mean_6h vs positional rolling(6).shift(1): max|diff| = 256.2

same column against a positional shift(24): max|diff| = 822.0
-> upstream lags are time-aligned; with 56k gaps the positional recipe would be wrong.


Note on what this proves: it verifies the columns are functions of the past
**as indexed**. Notebook 01 separately established that the underlying lags
are time-aligned, so "the past" here means real wall-clock past, not merely
earlier rows.

## 6. Leakage scan over the full candidate set

A feature that predicts the future better than the present value does is a
suspect. Persistence sets the bar: `corr(pm25, target)`.

In [9]:
candidate = pt.select_features(df, horizon=6)
print("candidate features for h=6:", len(candidate))

scan = pt.leakage_scan(df, candidate, "target_pm25_t_plus_6h", sample=200_000)
print("persistence correlation:", round(scan.attrs["persistence_corr"], 4))
print("\ntop 15 by correlation with the 6 h target:")
print(scan.head(15).to_string(index=False))

suspects = scan[scan["suspect"]]
print("\nsuspects (corr > 0.995):", len(suspects))
if len(suspects):
    print(suspects.to_string(index=False))

candidate features for h=6: 195


persistence correlation: 0.5676

top 15 by correlation with the 6 h target:
                  feature  corr_target  corr_vs_persistence  suspect
       pm25_roll_mean_24h      0.68195              0.11436    False
     pm25_roll_median_24h      0.67944              0.11185    False
       pm25_roll_mean_48h      0.67341              0.10582    False
       pm25_roll_mean_72h      0.66624              0.09866    False
      pm25_roll_mean_168h      0.64309              0.07550    False
       pm25_roll_mean_12h      0.61630              0.04871    False
station_baseline_by_month      0.60605              0.03847    False
        pm25_roll_p90_24h      0.59777              0.03018    False
        pm25_roll_mean_6h      0.59119              0.02361    False
        pm25_roll_min_24h      0.57463              0.00704    False
        pm25_roll_mean_3h      0.57141              0.00382    False
                     pm25      0.56759              0.00000    False
              pm25_lag_1h  

### Positive control

The scan is only worth running if it can actually catch something. Inject the
target as a feature and confirm it is flagged.

In [10]:
probe = df.head(200_000).copy()
probe["injected_leak"] = probe["target_pm25_t_plus_6h"]
control = pt.leakage_scan(probe, ["pm25", "injected_leak"], "target_pm25_t_plus_6h", sample=200_000)
print(control.to_string(index=False))
assert bool(control.loc[control["feature"] == "injected_leak", "suspect"].iloc[0])
print("\nPASS — the scan flags an injected leak and leaves persistence alone.")

      feature  corr_target  corr_vs_persistence  suspect
injected_leak      1.00000              0.36632     True
         pm25      0.63368              0.00000    False

PASS — the scan flags an injected leak and leaves persistence alone.


## 7. Save

In [11]:
out = cfg.feature_dir / "temporal_features.parquet"
df.to_parquet(out, index=False)

import joblib
joblib.dump(baselines, cfg.feature_dir / "station_baselines.joblib")

manifest = {
    "schema_version": cfg.schema_version,
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "rows": int(len(df)),
    "columns": int(df.shape[1]),
    "station_features_added": pt.FEATURE_COLUMNS_STATION,
    "baseline_fit_end": str(fit_end),
    "baseline_summary": baselines.to_dict(),
    "upstream_audit": audit.to_dict(orient="records"),
    "past_only_checks": checks,
    "leakage_suspects": suspects["feature"].tolist(),
    "fingerprint": pt.dataset_fingerprint(df[["timestamp_utc", "location_id", "pm25"]]),
}
(cfg.feature_dir / "temporal_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))

print("02 COMPLETE")
print("  ", out)
print("Next: 03_propagation_transport_features.ipynb")

02 COMPLETE
   /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast/data/propagation/processed/features/temporal_features.parquet
Next: 03_propagation_transport_features.ipynb


## Findings

**Most of what the plan asks for already exists.** 10 of 14 requested feature
families are complete in the upstream store: cyclical time, the full 9-deep
lag hierarchy (1/2/3/6/12/24/48/72/168 h), rolling mean/std/min/max/slope,
multi-scale regional fields (25/50/100 km), spatial dispersion, wind-aware
upwind PM2.5, and upwind fire. The plan was written against an earlier
version of the feature store and its sections 7-9, 11 and 14 are largely
satisfied. `fire_frp_sum_25km_24h` is "partial" only because upstream uses
5/10/20/50/100/150 km radii instead of the plan's 10/25/50 — a naming
mismatch, not a missing capability.

**Two families were genuinely missing and are added here:** day-of-week
cyclical encoding, and the whole of section 10 — station statistics
(`mean`/`std`/`p90`/`p95`), hour- and month-of-year baselines, and the
derived anomaly features. 11 new columns.

**Calendar columns are IST-local, not UTC.** Verified: upstream `hour` matches
the local hour on 100% of rows and the UTC hour on 0%. This is the right
convention for PM2.5 — the diurnal cycle tracks local traffic and
boundary-layer collapse — but it is a trap for anyone deriving a new cyclical
feature from `timestamp_utc`. `add_cyclical_time` reuses the existing
calendar columns, so the new `sin_dow`/`cos_dow` are on the same clock (100%
consistent).

**Station baselines are leakage-controlled.** Fitted on the 1,193,612 rows
before 2026-03-22 (the 70% cutoff) and stored with that cutoff in the
artifact. The tampering test — multiply every post-cutoff PM2.5 by 10 —
leaves `station_pm25_mean` bit-identical, which is the property that matters.

**The station-relative features are not obviously additive.** Correlation
with the 6 h residual: `pm25_vs_station_mean` -0.491, raw `pm25` -0.470,
`pm25_vs_station_hour` -0.443, `station_recent_anomaly` -0.421. All four sit
in the same band, so these are a re-parameterisation of mean reversion rather
than new information. Whether they earn their place is an empirical question
for notebook 05's ablation, not something to assert here.

**Past-only verification passes on every column tested**, and the recipes are
worth recording because two of them are not what you would guess:

| column | actual recipe | max abs diff |
|---|---|---|
| `pm25_lag_{1,24,168}h` | wall-clock lookup, not `shift(n)` | 0.00 |
| `pm25_roll_mean_{6,24}h` | time window, `closed="left"` | 2e-5 |
| `pm25_roll_std_6h` | time window, `closed="left"`, **ddof=0** | 3e-4 |

The same columns compared against the naive positional recipes differ by
**256 ug/m3** (rolling) and **822 ug/m3** (lag). The upstream implementation
is gap-aware and the naive reimplementation would have been wrong — which is
precisely why this notebook audits instead of rebuilding.

**Leakage scan: clean.** Persistence sets the bar at corr 0.568 with the 6 h
target. The top features are all longer-window rolling means (24 h at 0.682,
48 h at 0.673) — they beat persistence because they are smoother, not
because they see the future. Zero features exceed the 0.995 suspect
threshold. The positive control confirms the scan works: an injected copy of
the target is flagged at corr 1.000 while `pm25` is not.